# SÓSIA no Colab

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ridd1kulusC0d3r/SOSIA/blob/claude/laughing-davinci-bj48ot/notebooks/sosia_colab.ipynb)

Proteção de marca/pessoa contra perfis que se passam por ela — **OSINT passivo, explicável e mensurável**.

> **Uso responsável:** apenas dados públicos, sem login, sem contornar CAPTCHA/bloqueios, rate limit por host.
> Use para marca/pessoa que **você representa**. Prioridade ≠ veredito jurídico; **revise manualmente antes de denunciar**.
> As células marcadas **🌐 rede** fazem requisições reais; as demais rodam offline.

## 1. Instalação

In [ ]:
BRANCH = "claude/laughing-davinci-bj48ot"  #@param {type:"string"}
!pip -q install "git+https://github.com/Ridd1kulusC0d3r/SOSIA@{BRANCH}"
import sosia; print("SÓSIA", sosia.__version__)
# PDF é uma exportação opcional, não uma dependência do motor.
!pip -q install 'reportlab>=4'


## 2. Configure a marca (leia antes de preencher)

Cada campo altera uma parte diferente da investigação. Use **somente uma identidade que você representa**. Os valores de "Marca Zeta" são fictícios.

| Campo da tela | Por que preencher? | Exemplo |
|---|---|---|
| **BRAND** | Nome público usado no relatório, na interpretação das bios e na descoberta de nomes de exibição. | `Marca Zeta` |
| **HANDLE** | Username base **sem @** a partir do qual o motor gera variantes e mede semelhança. | `marcazeta` |
| **DOMAINS** | Domínios oficiais conhecidos, para contextualizar links em bios e sinalizar endereços parecidos. **Linkar o domínio oficial não prova vínculo.** | `marcazeta.com.br` |
| **OFFICIAL** | Perfis **realmente autorizados**, excluídos da geração e da sinalização (**allowlist**). Neste formulário, a lista vale para todas as plataformas. | `marcazeta, marcazeta_sac` |
| **EXTRA_AFFIXES** | Termos adicionais do seu setor que ajudam o gerador a testar combinações, como `marcazeta_clube`. **Não são perfis oficiais.** | `clube, cartao` |

**Formato:** para campos com vários valores, separe com vírgula. Em **DOMAINS**, informe só o domínio, sem `https://` ou caminhos. Em **HANDLE/OFFICIAL**, não use o símbolo `@`. Preencha OFFICIAL apenas com contas verificadas por sua organização. Se não souber um canal, deixe fora da allowlist em vez de presumir que é oficial.

> ⚠️ **Atenção ao escopo:** `OFFICIAL` no Colab é uma lista global (`"*"`). Para separar autorizados por plataforma, use `sosia.toml` com `[official]` e chaves como `instagram`, `x`, etc. Confira o [guia detalhado](https://github.com/Ridd1kulusC0d3r/SOSIA/blob/claude/laughing-davinci-bj48ot/docs/CONFIGURATION.md).


In [ ]:
#@title Marca / pessoa protegida
# BRAND: nome público da identidade que você representa; aparece no relatório e orienta a análise.
BRAND = "Marca Zeta"                  #@param {type:"string"}
# HANDLE: username de referência sem @; base para gerar as variantes.
HANDLE = "marcazeta"                  #@param {type:"string"}
# DOMAINS: domínios oficiais, separados por vírgula, sem https://.
DOMAINS = "marcazeta.com.br"          #@param {type:"string"}
# OFFICIAL: allowlist de contas comprovadamente autorizadas; aplicada a TODAS as plataformas neste Colab.
OFFICIAL = "marcazeta, marcazeta_sac"  #@param {type:"string"}
# EXTRA_AFFIXES: palavras específicas do setor para ampliar os usernames candidatos; não significam autorização.
EXTRA_AFFIXES = "clube, cartao"        #@param {type:"string"}

from sosia.model import Brand
_split = lambda s: [x.strip() for x in s.split(",") if x.strip()]
if not BRAND.strip() or not HANDLE.strip():
    raise ValueError("BRAND e HANDLE não podem ficar vazios.")
brand = Brand(name=BRAND.strip(), handle=HANDLE.strip().lstrip("@").lower(),
              domains=_split(DOMAINS),
              official={"*": {h.lower().lstrip("@") for h in _split(OFFICIAL)}},
              extra_affixes=[x.lower() for x in _split(EXTRA_AFFIXES)])
print("Marca:", brand.name)
print("Handle base:", brand.handle)
print("Domínios:", brand.domains)
print("Allowlist global:", sorted(brand.official["*"]))
print("Termos adicionais:", brand.extra_affixes)


## 3. Variantes de username (offline)
Ranqueadas por *massa de probabilidade por família* — veja `docs/DESIGN.md`. Os pesos são hipóteses; re-estime com `sosia fit-priors`.

In [ ]:
from sosia.variants import VariantEngine
variants = VariantEngine(brand.handle, extra_affixes=brand.extra_affixes).generate(exclude=brand.official["*"])
print(f"{len(variants)} variantes")
rows = [(v.value, ", ".join(sorted(v.families)), v.similarity) for v in variants[:40]]
try:
    import pandas as pd; display(pd.DataFrame(rows, columns=["variante", "famílias", "semelhança"]))
except ImportError:
    for r in rows: print(r)

## 4. Ler uma bio (offline)
Sinais agrupados, com negação e mitigadores. **CRÍTICO** exige evidência independente — uma palavra solta nunca basta.

In [ ]:
from sosia.profiles import Profile, find_links
from sosia.signals import analyze_bio
from sosia.scoring import assess
from sosia.textutil import similarity

def triage(username, display_name, bio):
    p = Profile(display_name=display_name, bio=bio or None, links=find_links(bio or ""))
    rep = analyze_bio(p, brand)
    a = assess(brand, "*", username, similarity(brand.handle, username), 0.95, p, rep)
    print(f"{username}: {a.priority.value} · abuso={a.abuse_level} · severidade={a.severity} · vínculo={a.affiliation}")
    for r in a.reasons: print("   -", r)

triage("marcazeta_suporte", "Marca Zeta Suporte", "Suporte oficial Marca Zeta. Chame no WhatsApp e faça o Pix da taxa de liberação.")
triage("marcazeta_loja", "Zeta Revenda", "Revenda autorizada. Aceitamos Pix e cartão. Atendimento via WhatsApp.")
triage("marcazeta_sac2", "Marca Zeta", "Atendimento oficial. Nunca pedimos senha ou Pix por aqui.")

## 5. 🌐 Calibrar e validar os conectores no *seu* ambiente
O resultado depende de IP/região. Só plataformas `trusted`/validadas entram na varredura; bloqueio, CAPTCHA e soft-404 viram **inconclusivo**, nunca ausência.

In [ ]:
PLATFORMS = "telegram, youtube, tiktok, linktree, x, mastodon, bluesky, devto, gitlab"  #@param {type:"string"}

from sosia.platforms import load_platforms
from sosia.net import UrlLibFetcher
from sosia.calibrate import calibrate, validate

plats = load_platforms(only=_split(PLATFORMS))
fetcher = UrlLibFetcher(min_interval=0.7, timeout=10)
usable = {}
for pid, p in plats.items():
    cal = calibrate(p, fetcher)
    val = validate(p, cal, fetcher) if cal.usable else None
    ok = cal.usable and (val is None or val.passed or cal.state.value != "differential")
    print(("OK " if ok else "-- "), f"{pid:<10} {cal.state.value:<20} {cal.reason}",
          f"| validação {val.known_hit}/{val.known_total} · {val.random_hit}/{val.random_total}" if val else "")
    if cal.usable: usable[pid] = p
print("\nutilizáveis:", list(usable))

## 6. 🌐 Varredura
Um pacote de evidências (`case.json`, `report.md`, CSV, STIX 2.1, `MANIFEST.sha256`) é gravado em `/content/runs`.

In [ ]:
BUDGET = 80  #@param {type:"integer"}
WATCH  = ""  #@param {type:"string"}

import time, pathlib
from sosia.scanner import Scanner, ScanConfig
from sosia.report import write_case, summary_counts, to_markdown, finding_line
from sosia.model import Existence

cfg = ScanConfig(budget=BUDGET, seed=1, extra_candidates=_split(WATCH), save_raw=False)
res = Scanner(brand, plats, fetcher, cfg, progress=print).run()
out = pathlib.Path("/content/runs") / f"{brand.handle}-{time.strftime('%Y%m%d-%H%M%S')}"
write_case(res, out)
t = res.stats["totals"]
print(f"\nrequisições={t['requests']} encontrados={t['found']} inconclusivos={t['inconclusive']} descartadas={t['platforms_discarded']}")
print("prioridades:", summary_counts(res))
for f in res.sorted_findings():
    if f.existence in (Existence.CONFIRMED, Existence.PROBABLE): print(finding_line(f))
print("\npacote:", out)

In [ ]:
from IPython.display import Markdown, display
display(Markdown(to_markdown(res)))

## 7. Exportar por escolha: HTML, PDF, CSV, JSON, Markdown, STIX ou ZIP

Depois que o scan terminar, selecione os formatos na próxima célula. O HTML é responsivo e pode ser aberto no navegador; o PDF é paginado e próprio para leitura/compartilhamento controlado.

**Arquivos individuais** acionam um download por formato. **Selecionados (ZIP)** contém só os formatos marcados. **Pacote completo (ZIP)** contém todos os arquivos do caso, inclusive evidências e o manifesto de hashes.

> **Privacidade:** pacotes de evidências podem conter bios, URLs e metadados públicos de terceiros. Revise o conteúdo antes de compartilhar. O manifesto `MANIFEST.sha256` cobre o pacote integral e inclui os novos HTML/PDF quando gerados. Um ZIP parcial não contém necessariamente todos os itens referenciados por esse manifesto.


In [ ]:
#@title Download personalizado de resultados
BAIXAR_PDF = True          #@param {type:"boolean"}
BAIXAR_HTML = True         #@param {type:"boolean"}
BAIXAR_CSV = True          #@param {type:"boolean"}
BAIXAR_JSON = True         #@param {type:"boolean"}
BAIXAR_MARKDOWN = False    #@param {type:"boolean"}
BAIXAR_JSONL = False       #@param {type:"boolean"}
BAIXAR_STIX = False        #@param {type:"boolean"}
BAIXAR_MANIFESTO = False   #@param {type:"boolean"}
INCLUIR_EVIDENCIAS = False #@param {type:"boolean"}
MODO_DOWNLOAD = "Selecionados (ZIP)"  #@param ["Selecionados (ZIP)", "Arquivos individuais", "Pacote completo (ZIP)"]

from pathlib import Path
import zipfile
from sosia.exports import export_reports

if "res" not in globals() or "out" not in globals() or not Path(out).exists():
    raise RuntimeError("Execute a célula de varredura antes de exportar.")

out = Path(out)
# Geramos somente os formatos adicionais desejados, exceto no pacote completo.
full = MODO_DOWNLOAD == "Pacote completo (ZIP)"
new_files = export_reports(
    res, out, html_report=BAIXAR_HTML or full, pdf_report=BAIXAR_PDF or full
)
files_map = {
    "report.pdf": BAIXAR_PDF, "report.html": BAIXAR_HTML,
    "findings.csv": BAIXAR_CSV, "case.json": BAIXAR_JSON,
    "report.md": BAIXAR_MARKDOWN, "findings.jsonl": BAIXAR_JSONL,
    "stix.json": BAIXAR_STIX, "MANIFEST.sha256": BAIXAR_MANIFESTO,
}
selected = [out / name for name, enabled in files_map.items() if enabled]
if INCLUIR_EVIDENCIAS:
    selected.extend(p for p in (out / "evidence").rglob("*") if p.is_file())
if full:
    selected = [p for p in out.rglob("*") if p.is_file() and p.suffix != ".zip"]
if not selected:
    raise ValueError("Marque pelo menos um formato ou selecione o pacote completo.")
selected = sorted(set(selected))
print("Arquivos escolhidos:", *[str(p.relative_to(out)) for p in selected], sep="\n • ")

try:
    from google.colab import files
except ImportError:
    files = None

if MODO_DOWNLOAD == "Arquivos individuais":
    for item in selected:
        if files is not None:
            files.download(str(item))
        else:
            print("Disponível em:", item)
else:
    destination = out.parent / (out.name + ("-completo.zip" if full else "-selecionados.zip"))
    with zipfile.ZipFile(destination, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for item in selected:
            archive.write(item, arcname=item.relative_to(out).as_posix())
    print("ZIP:", destination)
    if files is not None:
        files.download(str(destination))


## 8. Bench sintético (offline, ≈10 s)
Harness de regressão com verdade-terreno **simulada**: **não** são taxas de detecção do mundo real (ver `docs/BENCH_RESULTS.md`).

In [ ]:
from sosia.bench.runner import run_bench, render_markdown
from IPython.display import Markdown, display
bench = run_bench(seed=7, quick=True)
display(Markdown(render_markdown(bench)))

## 9. Medir a triagem em perfis **reais** rotulados (a prova de verdade)
Suba um JSONL (`username, display_name, bio, label` com `label` ∈ `scam|nonscam|fan|benign|reseller`, mais `brand/handle/domains`), rotulado por dois revisores.
Veja `examples/labeled_sample.jsonl`. Em linha de comando: `sosia eval rotulados.jsonl`.

In [ ]:
import json, urllib.request
SAMPLE = f"https://raw.githubusercontent.com/Ridd1kulusC0d3r/SOSIA/{BRANCH}/examples/labeled_sample.jsonl"
urllib.request.urlretrieve(SAMPLE, "/content/labeled_sample.jsonl")
!sosia eval /content/labeled_sample.jsonl